# Docker, Streamlit, FastAPI, Redis, Kafka: Orchestration & Interaction Demo

Ce notebook montre comment orchestrer et interagir avec Streamlit, FastAPI, Redis et Kafka dans un environnement Dockerisé. Chaque section fournit le code, les fichiers de configuration, et des exemples d'appels interactifs.

## 1. Set Up Docker Environment

Nous allons créer les fichiers nécessaires pour orchestrer plusieurs services :
- `Dockerfile` pour Streamlit
- `Dockerfile` pour FastAPI
- `docker-compose.yml` pour tout orchestrer (incluant Redis et Kafka)

**Exemple : docker-compose.yml**

In [ ]:
# docker-compose.yml
compose_yml = '''
version: "3.8"
services:
  streamlit:
    build: ./streamlit_app
    ports:
      - "8501:8501"
    depends_on:
      - redis
      - kafka
  fastapi:
    build: ./fastapi_app
    ports:
      - "8000:8000"
    depends_on:
      - redis
      - kafka
  redis:
    image: redis:7
    ports:
      - "6379:6379"
  zookeeper:
    image: bitnami/zookeeper:latest
    environment:
      - ALLOW_ANONYMOUS_LOGIN=yes
    ports:
      - "2181:2181"
  kafka:
    image: bitnami/kafka:latest
    environment:
      - KAFKA_BROKER_ID=1
      - KAFKA_ZOOKEEPER_CONNECT=zookeeper:2181
      - KAFKA_LISTENERS=PLAINTEXT://:9092
      - KAFKA_ADVERTISED_LISTENERS=PLAINTEXT://kafka:9092
      - ALLOW_PLAINTEXT_LISTENER=yes
    ports:
      - "9092:9092"
    depends_on:
      - zookeeper
'''
print(compose_yml)

## 2. Build and Run Streamlit App in Docker

Créez un dossier `streamlit_app/` avec :
- `Dockerfile`
- `app.py`

**Exemple : Dockerfile pour Streamlit**

In [ ]:
# streamlit_app/Dockerfile
streamlit_dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY app.py .
RUN pip install streamlit redis kafka-python requests
CMD ["streamlit", "run", "app.py", "--server.port=8501", "--server.address=0.0.0.0"]
'''
print(streamlit_dockerfile)

# streamlit_app/app.py
streamlit_app = '''
import streamlit as st
st.title("Streamlit in Docker!")
st.write("Hello from inside a Docker container.")
'''
print(streamlit_app)

**Build & Run :**
```bash
docker-compose build streamlit
# puis
docker-compose up streamlit
```
Accédez à l'app : http://localhost:8501

## 3. Build and Run FastAPI Service in Docker

Créez un dossier `fastapi_app/` avec :
- `Dockerfile`
- `main.py`

**Exemple : Dockerfile pour FastAPI**

In [ ]:
# fastapi_app/Dockerfile
fastapi_dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY main.py .
RUN pip install fastapi uvicorn redis kafka-python
CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"]
'''
print(fastapi_dockerfile)

# fastapi_app/main.py
fastapi_app = '''
from fastapi import FastAPI
app = FastAPI()

@app.get("/ping")
def ping():
    return {"ping": "pong"}
'''
print(fastapi_app)

**Build & Run :**
```bash
docker-compose build fastapi
# puis
docker-compose up fastapi
```
Accédez à l'API : http://localhost:8000/ping

## 4. Set Up Redis and Kafka with Docker Compose

Redis et Kafka sont déjà définis dans le `docker-compose.yml` ci-dessus. Pour lancer tous les services :

```bash
docker-compose up -d
```

Vérifiez que tout fonctionne :
- Redis : `docker-compose exec redis redis-cli ping` (doit répondre PONG)
- Kafka : `docker-compose exec kafka kafka-topics.sh --list --bootstrap-server localhost:9092`

## 5. Interact with FastAPI and Streamlit from Notebook

Utilisez `requests` pour appeler les endpoints exposés par FastAPI et Streamlit.

In [ ]:
import requests

# Appel FastAPI
resp = requests.get("http://localhost:8000/ping")
print("FastAPI /ping:", resp.json())

# Appel Streamlit (page HTML)
resp = requests.get("http://localhost:8501")
print("Streamlit status:", resp.status_code)

## 6. Publish and Consume Messages with Kafka from Notebook

Utilisez `kafka-python` pour publier et consommer des messages sur un topic Kafka.

In [ ]:
from kafka import KafkaProducer, KafkaConsumer
import time

# Publier un message
producer = KafkaProducer(bootstrap_servers='localhost:9092')
producer.send('demo-topic', b'Hello Kafka from notebook!')
producer.flush()

# Consommer un message
consumer = KafkaConsumer('demo-topic', bootstrap_servers='localhost:9092', auto_offset_reset='earliest', consumer_timeout_ms=1000)
for msg in consumer:
    print("Kafka received:", msg.value.decode())
    break

## 7. Store and Retrieve Data with Redis from Notebook

Utilisez `redis-py` pour stocker et lire des données dans Redis.

In [ ]:
import redis

r = redis.Redis(host='localhost', port=6379, db=0)
r.set('notebook-key', 'hello from notebook')
print('Redis get:', r.get('notebook-key').decode())

## 8. Exemple : Échange de messages entre Streamlit et FastAPI via Kafka

Dans cet exemple, Streamlit publie un message sur Kafka, FastAPI le consomme et expose le dernier message reçu via un endpoint. Le notebook vérifie l'échange bout-en-bout.

In [ ]:
# --- Streamlit publie un message sur Kafka ---
from kafka import KafkaProducer
producer = KafkaProducer(bootstrap_servers='localhost:9092')
producer.send('streamlit-to-fastapi', b'Hello from Streamlit!')
producer.flush()
print('Message envoyé sur Kafka.')

**Ajoutez ce code dans fastapi_app/main.py pour consommer Kafka et exposer le dernier message :**

In [ ]:
# fastapi_app/main.py (ajout)
from threading import Thread
from kafka import KafkaConsumer

last_msg = {'value': None}

def kafka_listener():
    consumer = KafkaConsumer('streamlit-to-fastapi', bootstrap_servers='kafka:9092', auto_offset_reset='latest')
    for msg in consumer:
        last_msg['value'] = msg.value.decode()

@app.on_event("startup")
def start_kafka():
    Thread(target=kafka_listener, daemon=True).start()

@app.get("/last-streamlit-msg")
def last_streamlit_msg():
    return {"msg": last_msg['value']}

In [ ]:
# --- Vérification notebook : FastAPI reçoit bien le message Streamlit ---
import time
for _ in range(10):
    resp = requests.get("http://localhost:8000/last-streamlit-msg")
    print("Dernier message reçu par FastAPI:", resp.json())
    if resp.json().get('msg'):
        break
    time.sleep(1)

## 9. Visualisation Streamlit en temps réel (via Redis)

Dans cet exemple, FastAPI écrit des données dans Redis, et Streamlit lit et affiche ces données en temps réel. Le notebook simule l'écriture, puis Streamlit affiche dynamiquement la valeur.

In [ ]:
# --- FastAPI écrit une valeur dans Redis ---
import redis
r = redis.Redis(host='localhost', port=6379, db=0)
r.set('streamlit-live', 'Valeur dynamique depuis FastAPI')
print('Valeur écrite dans Redis.')

**Ajoutez ce code dans `streamlit_app/app.py` pour afficher la valeur en temps réel :**

In [ ]:
# streamlit_app/app.py (ajout)
import streamlit as st
import redis
import time

r = redis.Redis(host='redis', port=6379, db=0)
st.title('Visualisation temps réel via Redis')

placeholder = st.empty()
for _ in range(100):
    val = r.get('streamlit-live')
    if val:
        placeholder.write(val.decode())
    time.sleep(1)

## 10. Dashboard multi-symboles et écriture concurrente (Redis + Streamlit)

Dans cet exemple, plusieurs producteurs (FastAPI, notebook, etc.) écrivent en parallèle des prix pour différents symboles dans Redis. Streamlit lit et affiche tous les symboles en temps réel sous forme de tableau ou graphique.

In [ ]:
# --- Plusieurs producteurs écrivent en parallèle dans Redis ---
import redis, random, time
r = redis.Redis(host='localhost', port=6379, db=0)
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']
for _ in range(10):
    for sym in symbols:
        price = round(random.uniform(100, 70000), 2)
        r.set(f'price:{sym}', price)
    time.sleep(1)
print('Prix multi-symboles écrits dans Redis.')

**Ajoutez ce code dans `streamlit_app/app.py` pour un dashboard multi-symboles temps réel :**

In [ ]:
# streamlit_app/app.py (dashboard multi-symboles)
import streamlit as st
import redis
import time

r = redis.Redis(host='redis', port=6379, db=0)
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT']
st.title('Dashboard multi-symboles temps réel')

placeholder = st.empty()
prices = {sym: [] for sym in symbols}
for _ in range(30):
    row = {}
    for sym in symbols:
        val = r.get(f'price:{sym}')
        price = float(val.decode()) if val else None
        prices[sym].append(price)
        row[sym] = price
    placeholder.table([row])
    time.sleep(1)

# Optionnel : afficher l'évolution sous forme de graphique
st.line_chart(prices)

## 11. Dashboard enrichi : multi-symboles, signaux, alertes visuelles

Ajoutez plus de symboles, affichez des signaux (ex : tendance, volatilité), et surlignez les alertes (ex : whale trade, drawdown) en couleur dans Streamlit.

In [ ]:
# --- Producteur enrichi : écrit prix, tendance, whale alert ---
import redis, random, time
r = redis.Redis(host='localhost', port=6379, db=0)
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'ARB/USDT', 'DOGE/USDT']
for _ in range(20):
    for sym in symbols:
        price = round(random.uniform(0.05, 70000), 2)
        trend = random.choice(['UP', 'DOWN', 'FLAT'])
        whale = price > 60000 and sym == 'BTC/USDT'
        r.set(f'price:{sym}', price)
        r.set(f'trend:{sym}', trend)
        if whale:
            r.set(f'alert:{sym}', f'WHALE TRADE {price}')
        else:
            r.delete(f'alert:{sym}')
    time.sleep(1)
print('Données enrichies écrites dans Redis.')

**Dashboard Streamlit enrichi : visualisation multi-symboles, signaux, alertes**

Ajoutez ce code dans `streamlit_app/app.py` pour afficher :
- Prix et tendance de chaque symbole
- Alerte visuelle (badge rouge) si whale trade détecté
- Tableau et graphique évolutif

In [ ]:
# streamlit_app/app.py (dashboard enrichi)
import streamlit as st
import redis
import time

r = redis.Redis(host='redis', port=6379, db=0)
symbols = ['BTC/USDT', 'ETH/USDT', 'SOL/USDT', 'BNB/USDT', 'ARB/USDT', 'DOGE/USDT']
st.title('Dashboard multi-symboles enrichi')

placeholder = st.empty()
prices = {sym: [] for sym in symbols}
trends = {sym: [] for sym in symbols}
alerts = {sym: [] for sym in symbols}

for _ in range(30):
    rows = []
    for sym in symbols:
        price = r.get(f'price:{sym}')
        trend = r.get(f'trend:{sym}')
        alert = r.get(f'alert:{sym}')
        price = float(price.decode()) if price else None
        trend = trend.decode() if trend else ''
        alert = alert.decode() if alert else ''
        prices[sym].append(price)
        trends[sym].append(trend)
        alerts[sym].append(alert)
        row = {
            'Symbole': sym,
            'Prix': price,
            'Tendance': trend,
            'Alerte': alert
        }
        rows.append(row)
    df = rows
    # Affichage tableau avec alertes visuelles
    st.write('---')
    for row in df:
        alert_color = 'red' if row['Alerte'] else 'green'
        st.markdown(f"**{row['Symbole']}** | Prix: {row['Prix']} | Tendance: {row['Tendance']} " +
                    (f"<span style='color:{alert_color};font-weight:bold'> {row['Alerte']} </span>" if row['Alerte'] else ''), unsafe_allow_html=True)
    st.line_chart({sym: prices[sym] for sym in symbols})
    time.sleep(1)
    placeholder.empty()